In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [13]:
url = "https://books.toscrape.com/catalogue/page-1.html"
headers = {"User-Agent": "Mozilla/5.0 (learning project)"}
response = requests.get(url, headers=headers, timeout=10)
print(response.status_code)

200


In [14]:
soup = BeautifulSoup(response.text, "html.parser")
books = soup.select("article.product_pod")
print(len(books))

20


In [15]:
book = books[0]
print(book.h3.a["title"])
print(book.select_one("p.price_color").text)
print(book.select_one("p.star-rating")["class"][1])
print(book.select_one("p.instock").text.strip())

A Light in the Attic
Â£51.77
Three
In stock


In [16]:
rows = []

for page in range(1, 6):  # pages 1 to 5
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"
    r = requests.get(url, headers=headers, timeout=10)
    if r.status_code != 200:
        break
    soup = BeautifulSoup(r.text, "html.parser")
    for book in soup.select("article.product_pod"):
        rows.append({
            "title": book.h3.a["title"],
            "price": book.select_one("p.price_color").text,
            "rating": book.select_one("p.star-rating")["class"][1],
            "availability": book.select_one("p.instock").text.strip(),
        })
    time.sleep(1)

df = pd.DataFrame(rows)
print(len(df))
df.head()

100


,title,price,rating,availability
0,A Light in the Attic,Â£51.77,Three,In stock
1,Tipping the Velvet,Â£53.74,One,In stock
2,Soumission,Â£50.10,One,In stock
3,Sharp Objects,Â£47.82,Four,In stock
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock


In [17]:
df["price"] = df["price"].str.replace("Â£", "", regex=False).astype(float)
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df["rating"] = df["rating"].map(rating_map)
df = df.drop_duplicates()
df.head()

,title,price,rating,availability
0,A Light in the Attic,51.77,3,In stock
1,Tipping the Velvet,53.74,1,In stock
2,Soumission,50.10,1,In stock
3,Sharp Objects,47.82,4,In stock
4,Sapiens: A Brief History of Humankind,54.23,5,In stock


In [18]:
print(df.shape)
print(df.describe())

(100, 4)
            price      rating
count  100.000000  100.000000
mean    34.560700    2.930000
std     14.638531    1.423149
min     10.160000    1.000000
25%     19.897500    2.000000
50%     34.775000    3.000000
75%     47.967500    4.000000
max     58.110000    5.000000


In [19]:
df.to_csv("books_dataset.csv", index=False)

from google.colab import files
files.download("books_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
!pip install openpyxl
df.to_excel("books_dataset.xlsx", index=False)

from google.colab import files
files.download("books_dataset.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>